# 🗓️ 24일차 스터디 노트북 — 힙 정렬 (선택 정렬의 진화형)

**오늘 범위**: 06-8 힙 정렬 — 힙과 완전 이진 트리 · 트리↔배열 대응 · 루트 삭제와 재구성 · 배열을 힙으로 만들기 · 실습 6-16 · 보충수업 6-5 `heapq`

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[판별]**

---

## 오늘의 한 문장

> **"선택 정렬은 최댓값을 찾는 데 O(n)이 걸린다. 그걸 O(log n)으로 줄이면 어떻게 될까?"**

교재 286p 첫 줄이 **"선택 정렬을 응용한 알고리즘인 힙 정렬"** 이야. 오늘 배우는 건 새 알고리즘이 아니라 **19일차 선택 정렬의 업그레이드판**이야. 딱 하나가 바뀌어:

| | 최댓값 찾기 | 전체 |
|---|---|---|
| 선택 정렬 (19일) | 남은 구간을 전부 훑음 → **O(n)** | O(n²) |
| **힙 정렬 (오늘)** | 루트만 보면 됨 → **O(1)**, 대신 재구성에 **O(log n)** | **O(n log n)** |

## 오늘의 진행 순서

**개념(1~4) → 트리↔배열(5~6) → 루트 삭제(7~9) → 힙 정렬 전체(10~12) → 힙 만들기(13~14) → 코드(15~19) → heapq(20~21)**

트리를 **직접 그려가며** 푸는 문제가 많아. 종이랑 펜 준비해.

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from typing import MutableSequence
import random, time, heapq, math

def show_tree(a, n=None, title=""):
    """배열을 완전 이진 트리 모양으로 출력"""
    if n is None: n = len(a)
    if n == 0:
        print(title, "(빈 트리)"); return
    levels = []
    i, w = 0, 1
    while i < n:
        levels.append(list(range(i, min(i + w, n))))
        i += w; w *= 2
    depth = len(levels)
    total = (2 ** (depth - 1)) * 4
    if title: print(title)
    for d, lv in enumerate(levels):
        slot = total // (2 ** d)
        line = ""
        for pos, idx in enumerate(lv):
            center = pos * slot + slot // 2
            s = str(a[idx])
            start = center - len(s) // 2
            line += " " * max(0, start - len(line)) + s
        print(line)
    print()

def is_heap(a, n=None):
    """부모 >= 자식 조건을 만족하는지 검사"""
    if n is None: n = len(a)
    for p in range(n):
        for c in (2*p+1, 2*p+2):
            if c < n and a[p] < a[c]:
                return False, (p, c)
    return True, None

# 교재 실습 6-16
def heap_sort(a: MutableSequence) -> None:
    """힙 정렬"""
    def down_heap(a: MutableSequence, left: int, right: int) -> None:
        """a[left]~a[right]를 힙으로 만들기"""
        temp = a[left]                              # 루트
        parent = left
        while parent < (right + 1) // 2:
            cl = parent * 2 + 1                     # 왼쪽 자식
            cr = cl + 1                             # 오른쪽 자식
            child = cr if cr <= right and a[cr] > a[cl] else cl   # 큰 값을 선택
            if temp >= a[child]:
                break
            a[parent] = a[child]
            parent = child
        a[parent] = temp

    n = len(a)
    for i in range((n - 1) // 2, -1, -1):           # 1단계: 배열을 힙으로
        down_heap(a, i, n - 1)
    for i in range(n - 1, 0, -1):                   # 2단계: 루트를 꺼내며 정렬
        a[0], a[i] = a[i], a[0]
        down_heap(a, 0, i - 1)

def down_heap(a, left, right):
    """단독으로도 쓸 수 있게 꺼내둔 버전"""
    temp = a[left]; parent = left
    while parent < (right + 1) // 2:
        cl = parent * 2 + 1; cr = cl + 1
        child = cr if cr <= right and a[cr] > a[cl] else cl
        if temp >= a[child]: break
        a[parent] = a[child]; parent = child
    a[parent] = temp

print("준비 완료 ✅")
show_tree([10, 9, 5, 8, 3, 2, 4, 6, 7, 1], title="예시 — 교재 [그림 6-33]의 힙:")

---
# 🔁 [Remind] 워밍업 — 되감기

오늘은 **19일차**와 **23일차**를 동시에 소환해.

### R-1. 🟢 [설명] 19일차 선택 정렬, 다시

```python
for i in range(n - 1):
    m = i
    for j in range(i + 1, n):     # ← 여기
        if a[j] < a[m]: m = j
    a[i], a[m] = a[m], a[i]
```

- 안쪽 `for`는 뭘 하는 거지? 한 번 돌 때 비교를 몇 번 해?
- 전체 시간 복잡도가 O(n²)인 이유를 이 안쪽 `for`로 설명해봐.
- 23일차 17번 실측에서 **거의 정렬된 데이터를 줘도 선택 정렬만 빨라지지 않았어.** 왜였지?
- 💡 오늘의 질문: **"남은 구간의 최댓값을 O(n)보다 빨리 찾을 수는 없을까?"**

### R-2. 🟡 [설명] 23일차 병합 정렬과 비교할 것

| 항목 | 병합 (23일) | 힙 (오늘, 예측) |
|---|---|---|
| 추가 메모리 | O(n) | ①____ |
| 안정성 | ✅ | ②____ |
| 최악 시간 | O(n log n) | ③____ |

- ①~③을 **예측만** 해두고, 21번에서 답을 맞춰봐.
- 힌트: 힙 정렬은 배열 안에서 **자리를 바꿔가며** 정렬해.

*(여기에 답 작성)*

---
# 🌳 PART 1 — 힙이란 무엇인가 (1~4번)

> 코드는 아직 안 봐. **트리부터** 확실히 잡고 간다.

### 1. 🟢 [설명] 힙의 두 가지 조건

교재 286p: **힙(heap)은 '부모의 값이 자식의 값보다 항상 크다'는 조건을 만족하는 완전 이진 트리입니다.**

이 한 문장에 **조건이 두 개** 들어 있어. 분리해서 적어봐.

- **조건 A (모양)**: ①________________
- **조건 B (값)**: ②________________

그리고 각각에 대해:
- 조건 B에서 "크다"는 **엄격하게 더 크다**는 뜻일까? 교재가 뒤이어 뭐라고 하지? (같아도 되나?)
- 부모 ≥ 자식이면 **가장 큰 값은 어디에** 있어? 이걸 뭐라고 부르지?
- 그럼 **가장 작은 값**은 어디에 있을까? 루트처럼 한 곳으로 특정할 수 있어?

💡 `heap`은 "쌓아 놓음", "쌓아 놓은 더미"라는 뜻이야. 이름이 구조를 말해주고 있어.

*(여기에 답 작성)*

### 2. 🟢 [판별] 이 트리는 힙인가?

각각에 대해 **힙이면 O, 아니면 X와 이유**를 적어봐. 이유는 **"모양이 틀렸다"인지 "값 관계가 틀렸다"인지** 구분해서.

**(가)**
```
        9
      /   \
     7     8
    / \
   6   5
```

**(나)**
```
        4
      /   \
     6     8
    / \   /
   7   5 9
```
*(교재 [그림 6-32] ⓐ)*

**(다)**
```
        9
      /   \
     7     8
    / \   /
   6   5 4
```
*(교재 [그림 6-32] ⓑ)*

**(라)**
```
        9
      /   \
     8     7
        /  \
       6    5
```

**(마)**
```
        9
      /   \
     3     8
    / \   / \
   1   2 7   6
```

| | 힙? | 이유 |
|---|---|---|
| (가) | | |
| (나) | | |
| (다) | | |
| (라) | | |
| (마) | | |

💡 (라)를 특히 잘 봐. 교재 286p: **"'완전'은 부모는 왼쪽 자식부터 추가하여 모양을 유지한다는 뜻입니다."**

*(답을 적은 뒤 아래 셀로 확인)*

In [ ]:
cases = {
    "(가)": [9, 7, 8, 6, 5],
    "(나)": [4, 6, 8, 7, 5, 9],
    "(다)": [9, 7, 8, 6, 5, 4],
    "(마)": [9, 3, 8, 1, 2, 7, 6],
}
for name, arr in cases.items():
    ok, why = is_heap(arr)
    msg = "힙 ✅" if ok else f"힙 아님 ❌ (a[{why[0]}]={arr[why[0]]} < a[{why[1]}]={arr[why[1]]})"
    print(f"{name} {arr} → {msg}")

print("\n(라)는 배열로 표현할 수조차 없어. 왜인지 6번에서 확인!")

### 3. 🟡 [설명] 힙은 "부분 순서 트리"다

교재 287p: **"힙에서 부모와 자식 관계는 일정하지만 형제 사이의 대소 관계는 일정하지 않습니다."**

[그림 6-33]의 힙을 봐:
```
            10
          /    \
         9      5
        / \    / \
       8   3  2   4
      / \  /
     6  7 1
```

- 형제인 `8`과 `3`: 작은 쪽이 **왼쪽/오른쪽** 중 어디에 있어?
- 형제인 `6`과 `7`: 작은 쪽은 어디에?
- 두 답이 다르지? 그래서 힙을 **부분 순서 트리(partial ordered tree)** 라고 불러.
- **이게 왜 중요할까?** 힙을 완성해도 배열은 여전히 정렬되어 있지 않아. 아래 셀에서 확인해봐.
- 💡 그럼 힙은 뭐에 쓸모 있는 거야? **"전체 정렬"은 안 되지만 "최댓값 하나"는 O(1)에 알 수 있다.** 이게 오늘의 전부야.

*(답을 적은 뒤 실행)*

In [ ]:
h = [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
ok, _ = is_heap(h)
print("힙인가?", ok)
print("배열   :", h)
print("정렬됨?", h == sorted(h, reverse=True))
print("\n→ 힙이지만 정렬은 안 되어 있다!")
print(f"최댓값은 항상 a[0] = {h[0]}  ← 이것만 O(1)에 보장")
print(f"최솟값은? {min(h)} ... 어디 있는지 알 수 없다 (index {h.index(min(h))})")

### 4. 🟢 [계산] 부모와 자식의 인덱스 공식

교재 287p. 힙을 배열에 저장하면 인덱스 사이에 규칙이 생겨.

```
원소 a[i]에서
  · 부모      : a[ ①______ ]
  · 왼쪽 자식 : a[ ②______ ]
  · 오른쪽 자식: a[ ③______ ]
```

[그림 6-33]의 배열 `[10, 9, 5, 8, 3, 2, 4, 6, 7, 1]` 로 검산해봐.

| i | a[i] | 부모 인덱스 | 부모 값 | 왼쪽 자식 | 오른쪽 자식 |
|---|---|---|---|---|---|
| 3 | 8 | ④ | ⑤ | ⑥ | ⑦ |
| 2 | 5 | ⑧ | ⑨ | ⑩ | ⑪ |
| 4 | 3 | | | ⑫ (있나?) | ⑬ (있나?) |

- `i = 0`(루트)에 이 부모 공식을 넣으면 뭐가 나와? 문제가 없을까?
- 자식이 **없는** 노드는 어떻게 판별해? (힌트: 인덱스가 배열 범위를 넘어가면)

*(답을 적은 뒤 확인)*

In [ ]:
a = [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
n = len(a)
print("i  a[i]  부모      왼쪽자식   오른쪽자식")
for i in range(n):
    par = f"a[{(i-1)//2}]={a[(i-1)//2]}" if i > 0 else "(없음-루트)"
    cl, cr = 2*i+1, 2*i+2
    L = f"a[{cl}]={a[cl]}" if cl < n else "(없음)"
    R = f"a[{cr}]={a[cr]}" if cr < n else "(없음)"
    print(f"{i}  {a[i]:3d}   {par:12s} {L:10s} {R}")

print(f"\n⚠️ i=0에 부모 공식을 넣으면 (0-1)//2 = {(0-1)//2}  ← 파이썬 바닥 나눗셈!")
print("   음수 인덱스는 파이썬에서 '뒤에서부터'를 뜻해서 조용히 엉뚱한 값이 나온다.")

---
# 🔄 PART 2 — 트리 ↔ 배열 (5~6번)

> 힙의 진짜 마법은 **"트리인데 배열이다"** 라는 데 있어. 포인터도, 노드 객체도 필요 없어.

### 5. 🟢 [빈칸] 트리 → 배열

교재 287p: **"가장 위쪽에 있는 루트를 a[0]에 저장합니다. 그리고 한 단계 아래에서 왼쪽 원소에서 오른쪽 원소로 따라갑니다."**

즉 **위에서 아래로, 왼쪽에서 오른쪽으로** 순서대로 담아. (이걸 **레벨 순회**라고 해)

**(가)** 다음 트리를 배열로 바꿔봐.
```
            10
          /    \
         9      5
        / \    / \
       8   3  2   4
      / \  /
     6  7 1
```
```
 0   1   2   3   4   5   6   7   8   9
[__, __, __, __, __, __, __, __, __, __]
```

**(나)** 이번엔 다른 트리:
```
            7
          /   \
         5     6
        / \   /
       2   4 3
```
```
[__, __, __, __, __, __]
```

- (나)는 힙이야? 부모≥자식을 전부 확인해봐.
- 트리에 노드가 **6개**인데 배열 크기도 **6**이야. 낭비되는 칸이 하나도 없지? **"완전" 이진 트리이기 때문**이야. 만약 2번 문제의 (라)처럼 중간이 빈 트리를 배열에 담으면 어떻게 될까?

*(답을 적은 뒤 실행)*

In [ ]:
print("(가)")
show_tree([10, 9, 5, 8, 3, 2, 4, 6, 7, 1])
print("배열:", [10, 9, 5, 8, 3, 2, 4, 6, 7, 1])

print("\n(나)")
show_tree([7, 5, 6, 2, 4, 3])
print("배열:", [7, 5, 6, 2, 4, 3], "→ 힙?", is_heap([7, 5, 6, 2, 4, 3])[0])

print("\n⚠️ 만약 (라)처럼 왼쪽 자식이 비어 있다면?")
print("   [9, 8, 7, None, None, 6, 5] 처럼 빈 칸을 넣어야 하고,")
print("   깊이가 깊어질수록 낭비되는 칸이 기하급수로 늘어난다.")

### 6. 🟡 [빈칸] 배열 → 트리 (노드 채우기)

이번엔 반대 방향이야. 배열을 보고 **트리의 빈 노드를 채워봐.**

**(가)** `a = [9, 8, 6, 7, 5, 4, 2, 3, 1]`
```
            (①)
          /     \
        (②)     (③)
        /  \    /   \
      (④)  (⑤) (⑥)  (⑦)
      / \
    (⑧) (⑨)
```

**(나)** `a = [8, 7, 5, 3, 6, 2, 4]`
```
            (①)
          /     \
        (②)     (③)
        /  \    /   \
      (④)  (⑤) (⑥)  (⑦)
```
- (나)는 힙이야? 아니라면 **어느 부모-자식 쌍**이 조건을 깨?

**(다) 역방향 사고**: 어떤 힙의 배열이 `[?, ?, ?, ?, ?]` 5칸인데, **`a[4]`의 부모는 누구**야? 그리고 `a[1]`의 자식은?

*(답을 적은 뒤 실행)*

In [ ]:
for name, arr in (("(가)", [9, 8, 6, 7, 5, 4, 2, 3, 1]),
                  ("(나)", [8, 7, 5, 3, 6, 2, 4])):
    ok, why = is_heap(arr)
    show_tree(arr, title=f"{name} {arr}")
    if ok:
        print("  → 힙 ✅\n")
    else:
        p, c = why
        print(f"  → 힙 아님 ❌: a[{p}]={arr[p]} < a[{c}]={arr[c]}\n")

print("(다) 5칸 배열에서")
print(f"  a[4]의 부모 = a[{(4-1)//2}]")
print(f"  a[1]의 자식 = a[{2*1+1}], a[{2*1+2}]")

---
# ⬇️ PART 3 — 루트를 삭제한 힙의 재구성 (7~9번)

> 힙 정렬의 심장이야. 이 동작 하나만 이해하면 나머지는 반복일 뿐이야.
> 교재 288~289p [그림 6-34].

### 7. 🟢 [빈칸] 그림 6-34를 직접 그려보기

시작 힙 (그림 6-33과 같음):
```
            10
        /        \
       9          5
     /   \      /   \
    8     3    2     4
   / \   /
  6   7 1
```
배열: `[10, 9, 5, 8, 3, 2, 4, 6, 7, 1]`

**ⓐ 루트 10을 꺼내고, 마지막 원소를 루트로 옮긴다**
```
           (①)
        /        \
       9          5
     /   \      /   \
    8     3    2     4
   / \
  6   7
```
- ① = ____   (배열의 **마지막 원소**가 뭐였지?)
- 이때 트리에서 **1이 있던 자리는 어떻게 될까?**

**ⓑ 루트의 두 자식은 9와 5. 둘 중 큰 값과 교환**
```
           (②)
        /        \
      (③)         5
     /   \      /   \
    8     3    2     4
   / \
  6   7
```
- ② = ____ , ③ = ____

**ⓒ 다시 그 자리의 두 자식은 8과 3. 큰 값과 교환**
```
            9
        /        \
      (④)         5
     /   \      /   \
    (⑤)   3    2     4
   / \
  6   7
```
- ④ = ____ , ⑤ = ____

**ⓓ 다시 그 자리의 두 자식은 6과 7. 큰 값과 교환**
```
            9
        /        \
       8          5
     /   \      /   \
    (⑥)   3    2     4
   / \
  6  (⑦)
```
- ⑥ = ____ , ⑦ = ____

**최종 배열**: `[__, __, __, __, __, __, __, __, __]`

- 교재 289p: **"1은 가장 아래쪽인 리프(leaf)의 위치까지 이동했습니다. 하지만 원소를 항상 끝까지 이동시킬 필요는 없습니다."** 어떤 경우에 중간에 멈출까?
- 이번 예시에서 `1`이 이동한 **단계 수**는 몇 번이야? 트리의 높이와 비교하면?

*(답을 적은 뒤 실행)*

In [ ]:
h = [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
show_tree(h, title="시작 힙")

root = h[0]
h[0] = h[-1]; h = h[:-1]
print(f"ⓐ 루트 {root} 꺼냄 → 마지막 원소 {root if False else h[0]}을 루트로")
show_tree(h)

temp = h[0]; parent = 0; right = len(h) - 1
labels = "ⓑⓒⓓⓔ"; si = 0
while parent < (right + 1) // 2:
    cl = parent * 2 + 1; cr = cl + 1
    child = cr if cr <= right and h[cr] > h[cl] else cl
    if temp >= h[child]:
        print(f"   → 자식 {h[child]} 보다 {temp} 가 크거나 같으므로 멈춤")
        break
    print(f"{labels[si]} a[{parent}] 자리에 자식 a[{child}]={h[child]} 를 끌어올림")
    h[parent] = h[child]; parent = child; si += 1
h[parent] = temp
show_tree(h, title=f"최종 (원소 {temp}는 a[{parent}]에 안착)")
print("배열:", h, "| 힙인가?", is_heap(h)[0])

### 8. 🟡 [설명] 왜 하필 "마지막 원소"를 루트로 올릴까

교재 288p: **"비어 있는 루트 위치에 힙의 마지막 원소인 1을 이동합니다. 이때 이동한 1 이외의 원소는 힙 상태를 유지합니다."**

- 루트가 비었으면 그냥 **자식 중 큰 놈을 올리면** 안 될까? 그렇게 하면 무슨 문제가 생기지?
  💡 힌트: 그 자식 자리가 또 비고, 또 그 자식이 올라오고… 마지막엔 **트리 중간 어딘가에 구멍**이 남아. 그럼 **"완전" 이진 트리**가 깨지지?
- 마지막 원소를 쓰면 왜 모양이 유지될까?
- 교재는 **"이동한 1 이외의 원소는 힙 상태를 유지합니다"** 라고 해. 왜 그런지 설명해봐. (마지막 원소를 뽑아낸 자리는 리프였으니까…)
- 그래서 우리가 고쳐야 할 건 **딱 하나의 원소뿐**이고, 그걸 아래로 내리는 게 `down_heap`이야.

*(여기에 답 작성)*

### 9. 🟡 [설명] 왜 "큰 자식"과 교환할까

교재 288p ⓑ: **"1, 9, 5 가운데 최댓값이 가장 위쪽에 위치해야 합니다. '부모의 값 ≥ 자식의 값'이라는 힙의 조건이 성립하려면 두 자식을 비교하여 큰 값인 왼쪽 자식 9와 교환합니다."**

- 만약 **작은 자식(5)** 과 교환하면 어떻게 될까? 교환 후 트리를 그려보고, **어느 부모-자식 쌍이 깨지는지** 찾아봐.
```
            5
        /        \
       9          1
```
- 그래서 코드에 이 줄이 있는 거야:
```python
child = cr if cr <= right and a[cr] > a[cl] else cl   # 큰 값을 선택
```
  이 조건문을 **말로 풀어서** 설명해봐. `cr <= right` 는 왜 필요하지?
- 만약 두 자식의 값이 **같다면**? 어느 쪽이 선택돼? 그게 문제가 될까?

*(답을 적은 뒤 실행)*

In [ ]:
# 작은 자식과 교환하면?
bad = [5, 9, 1, 8, 3, 2, 4, 6, 7]
show_tree(bad, title="1과 5를 교환했다면 (잘못된 선택)")
ok, why = is_heap(bad)
p, c = why
print(f"→ 힙 깨짐 ❌: a[{p}]={bad[p]} < a[{c}]={bad[c]}")

good = [9, 1, 5, 8, 3, 2, 4, 6, 7]
show_tree(good, title="1과 9를 교환했다면 (올바른 선택)")
ok, why = is_heap(good)
print("→ 루트는 해결됨. 아직 아래가 남았지만 위쪽은 정상:",
      f"a[0]={good[0]} >= a[1]={good[1]}, a[2]={good[2]} ✅")

---
# 🔁 PART 4 — 힙 정렬의 전체 흐름 (10~12번)

> 교재 290~291p [그림 6-35]. 여기가 오늘의 하이라이트야.
> **배열과 트리를 동시에** 따라가면서 그려봐.

### 10. 🟡 [빈칸] 배열과 트리를 나란히 — 그림 6-35 재현

시작 (이미 힙 상태):
```
배열: [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
       └─────── 아직 정렬 안 됨(힙) ───────┘   정렬 완료: (없음)
```

**ⓐ 루트 `a[0]=10`과 맨 끝 `a[9]=1`을 교환**
```
배열: [①, 9, 5, 8, 3, 2, 4, 6, 7, ②]
       └───── 힙으로 다시 만들 범위 a[0]~a[8] ─────┘  정렬완료: [②]
```
- ① = ____ , ② = ____

**ⓑ `a[0]~a[8]`을 다시 힙으로 만든 뒤, 루트와 `a[8]`을 교환**

다시 힙으로 만든 결과 트리:
```
            (③)
        /        \
      (④)         5
     /   \      /   \
    7     3    2     4
   / \
  6   1
```
- ③ = ____ , ④ = ____
- 교환 후 배열: `[__, __, 5, 7, 3, 2, 4, 6, ⑤, 10]`  → ⑤ = ____
- 정렬 완료 구간: `[__, 10]`

**ⓒ~ⓔ** 같은 과정을 반복하면 배열 **뒤쪽부터** 큰 값이 차곡차곡 쌓여.
- ⓒ 끝난 뒤 정렬 완료 구간: `[__, __, 10]`
- ⓓ 끝난 뒤: `[__, __, __, 10]`
- ⓔ 끝난 뒤: `[__, __, __, __, 10]`

*(직접 채운 뒤 아래 셀로 대조)*

In [ ]:
a = [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
n = len(a)
labels = "ⓐⓑⓒⓓⓔⓕⓖⓗⓘ"
print(f"시작: {a}   (이미 힙)\n")

for step, i in enumerate(range(n - 1, 0, -1)):
    a[0], a[i] = a[i], a[0]
    print(f"{labels[step]} a[0]={a[i]} 와 a[{i}] 교환 → {a}")
    down_heap(a, 0, i - 1)
    heap_part = a[:i]
    print(f"   a[0]~a[{i-1}] 재구성 → {a}")
    print(f"   힙 부분 {heap_part}  |  정렬 완료 {a[i:]}")
    if step < 3:
        show_tree(a, n=i, title=f"   (힙 부분만 트리로)")
    print()

print("최종:", a)

### 11. 🟢 [설명] 4단계 알고리즘

교재 290p가 정리한 순서야. 빈칸을 채워봐.

```
1. i값을 ①______ 로 초기화합니다.
2. ②__________ 를 교환합니다.
3. a[0], a[1], ..., a[i-1]을 ③__________ 만듭니다.
4. i값을 ④______ 시켜 0이 되면 종료합니다. 그렇지 않으면 ⑤번으로 돌아갑니다.
```

- 이걸 파이썬 `for` 문 **두 줄**로 옮기면? (10번 셀에 답이 있어)
- 교재 290p 마지막 문장: **"배열의 처음 상태가 힙의 요구 사항을 만족하지 않을 수도 있습니다. 따라서 이 순서를 적용하기 전에 배열을 반드시 힙으로 만들어야 합니다."**
  → 이게 **PART 5(13~14번)** 에서 할 일이야. 힙 정렬은 총 **몇 단계**로 구성되는 거지?

*(여기에 답 작성)*

### 12. 🟡 [설명] 왜 정렬 완료 구간이 "뒤에서부터" 자랄까

10번 실행 결과를 보면 정렬된 부분이 배열 **오른쪽 끝에서 왼쪽으로** 자라나.

- 루트는 **최댓값**이야. 최댓값이 최종적으로 가야 할 자리는 배열의 **어디**지?
- 그래서 루트를 "꺼낸다"고 안 하고 **"맨 끝 원소와 교환한다"** 고 해. 왜 이게 똑똑한 방법이야? (힌트: 추가 배열이 필요한가?)
- **19일차 선택 정렬과 나란히 놓고 비교**해봐:

| | 매 라운드 하는 일 | 최댓값 찾기 비용 | 총 라운드 수 | 전체 |
|---|---|---|---|---|
| 선택 정렬 | 남은 구간에서 최댓값 찾아 맨 뒤와 교환 | ① | ② | ③ |
| 힙 정렬 | 루트(=최댓값)를 맨 뒤와 교환 후 재구성 | ④ | ⑤ | ⑥ |

- 교재 296p: **"단순 선택 정렬에서 최댓값인 원소를 선택하는 시간 복잡도는 O(n)이지만, 힙 정렬에서 다시 힙으로 만드는 작업의 시간 복잡도는 O(log n)입니다."**
  → `down_heap`이 왜 O(log n)이지? (한 번 내려갈 때마다 남은 범위가 어떻게 되지?)
- 교재는 이 동작이 **"이진 검색과 비슷하다"** 고 해. 어떤 점에서?

*(답을 적은 뒤 실행)*

In [ ]:
# down_heap이 몇 단계 내려가는지 측정
STEPS = [0]
def down_heap_count(a, left, right):
    temp = a[left]; parent = left
    while parent < (right + 1) // 2:
        cl = parent * 2 + 1; cr = cl + 1
        child = cr if cr <= right and a[cr] > a[cl] else cl
        if temp >= a[child]: break
        a[parent] = a[child]; parent = child
        STEPS[0] += 1
    a[parent] = temp

def heap_sort_count(a):
    n = len(a)
    for i in range((n - 1) // 2, -1, -1): down_heap_count(a, i, n - 1)
    for i in range(n - 1, 0, -1):
        a[0], a[i] = a[i], a[0]; down_heap_count(a, 0, i - 1)

random.seed(1)
print("n        총 이동단계   n*log2(n)")
for n in (100, 1000, 10000, 100000):
    STEPS[0] = 0
    heap_sort_count([random.randint(0, 10**6) for _ in range(n)])
    print(f"{n:7d}  {STEPS[0]:10d}   {n*math.log2(n):12.0f}")
print("\n→ 총 이동 단계가 n log n 에 비례하며 자란다")

---
# 🏗️ PART 5 — 배열을 힙으로 만들기 (13~14번)

> 11번에서 확인했듯 **정렬 전에 배열을 힙으로 만드는 준비 단계**가 필요해.
> 교재 292~293p [그림 6-36], [그림 6-37].

### 13. 🟡 [빈칸] 그림 6-37 — 아래에서 위로

시작 트리 (무작위 상태, 힙 아님):
```
             1
        /        \
       3          5
     /   \      /   \
    7     9    2     4
   / \   /
  6   8 10
```
배열: `[1, 3, 5, 7, 9, 2, 4, 6, 8, 10]`  (n = 10)

교재 292p: **"가장 아랫부분의 작은 서브트리부터 상향식(bottom-up)으로 진행하여 전체 배열을 힙으로 만들 수 있습니다."**

**ⓐ 마지막 서브트리 `(9, 10)`에 주목** — 부모 `9`, 자식 `10`
- 부모 `9` < 자식 `10` 이므로 교환 → 배열: `[1, 3, 5, 7, ①, 2, 4, 6, 8, ②]`
- ① = ____ , ② = ____

**ⓑ 왼쪽 서브트리 `(7, 6, 8)`에 주목** — 부모 `7`, 자식 `6`과 `8`
- 큰 자식은 ③____ 이고 부모보다 크므로 교환 → 배열: `[1, 3, 5, ④, 10, 2, 4, 6, ⑤, 9]`
- ③ = ____ , ④ = ____ , ⑤ = ____

**ⓒ 한 단계 위, 오른쪽 서브트리 `(5, 2, 4)`**
- 부모 `5`가 두 자식보다 크네? → **⑥________** (아무 일도 안 일어남)

**ⓓ 왼쪽 서브트리 `(3, 8, 10)`**
- 큰 자식 ⑦____ 과 교환. 그런데 내려간 `3`이 또 자식을 갖고 있어! → **한 번 더** 내려가.
- 결과 배열: `[1, ⑧, 5, 8, 9, 2, 4, 6, 7, ⑨]`

**ⓔ 마지막으로 루트 `1`**
- 아래로 계속 내려보내면 → 최종 배열: `[__, __, __, __, __, __, __, __, __, __]`

- 🎯 **최종 결과가 5번 문제의 트리(그림 6-33)와 같아진다!** 확인해봐.

*(답을 적은 뒤 실행)*

In [ ]:
b = [1, 3, 5, 7, 9, 2, 4, 6, 8, 10]
show_tree(b, title="ⓐ 시작 (힙 아님)")
n = len(b)
labels = "ⓐⓑⓒⓓⓔ"
for si, i in enumerate(range((n - 1) // 2, -1, -1)):
    before = b[:]
    down_heap(b, i, n - 1)
    changed = "" if before != b else "   ← 이미 힙이라 변화 없음"
    print(f"{labels[si]} i={i} (부모 a[{i}]={before[i]}, 자식 a[{2*i+1}]"
          + (f", a[{2*i+2}]" if 2*i+2 < n else "") + f"): {b}{changed}")
show_tree(b, title="\n완성된 힙")
print("그림 6-33의 힙과 같은가?", b == [10, 9, 5, 8, 3, 2, 4, 6, 7, 1])

### 14. 🔴 [설명] 왜 `(n-1)//2` 부터, 왜 역순일까

```python
for i in range((n - 1) // 2, -1, -1):
    down_heap(a, i, n - 1)
```

세 가지를 따져보자.

**(1) 시작점이 왜 `(n-1)//2` 야?**
- n = 10일 때 `(10-1)//2 = 4`. `a[4]`는 트리에서 어떤 위치야?
- `a[5]`부터 `a[9]`까지는 **자식이 있어?** (4번의 공식으로 `2*5+1 = 11 >= 10` 확인)
- 즉 `(n-1)//2`는 **자식을 가진 마지막 노드**야. 리프는 그 자체로 이미 힙이니 손댈 필요가 없지.
- 💡 다른 표현: `n//2 - 1`도 같은 값일까? n이 짝수/홀수일 때 각각 확인해봐.

**(2) 왜 역순(`-1` 스텝)이야?**
- `down_heap(a, i, ...)`은 **"`a[i]` 아래는 이미 힙"** 이라고 가정하고 동작해 (교재 295p: "a[left] 이외에는 모두 힙 상태라고 가정하고").
- 만약 `i = 0`부터 정순으로 돌면 이 가정이 성립할까?
- 아래에서 위로 올라가면 왜 항상 성립하지?

**(3) 시간 복잡도 함정** 🔥
- 노드가 n개, `down_heap`이 O(log n)이니 총 **O(n log n)** 일 것 같지? 그런데 실제로는 **O(n)** 이야.
- 힌트: **리프에 가까운 노드일수록 내려갈 거리가 짧아.** 전체 노드의 절반은 리프(0단계), 4분의 1은 1단계만 내려가고…
- 아래 실험이 실측으로 보여줘.

*(답을 적은 뒤 실행)*

In [ ]:
print("(1) 자식이 있는 마지막 노드")
for n in (10, 11, 7, 8):
    last_parent = (n - 1) // 2
    print(f"  n={n:3d}: (n-1)//2 = {last_parent}, n//2-1 = {n//2-1}, "
          f"a[{last_parent}]의 왼쪽자식 = a[{2*last_parent+1}] "
          f"({'존재' if 2*last_parent+1 < n else '없음'})")

print("\n(2) 정순으로 돌면?")
b1 = [1, 3, 5, 7, 9, 2, 4, 6, 8, 10]
b2 = b1[:]
for i in range((len(b2) - 1) // 2, -1, -1): down_heap(b2, i, len(b2) - 1)
print("  역순(교재):", b2, "→ 힙?", is_heap(b2)[0])
b3 = b1[:]
for i in range(0, (len(b3) - 1) // 2 + 1): down_heap(b3, i, len(b3) - 1)
print("  정순      :", b3, "→ 힙?", is_heap(b3)[0])

print("\n(3) 힙 만들기 1단계의 실제 이동 횟수")
BUILD = [0]
def dh_count(a, left, right):
    temp = a[left]; parent = left
    while parent < (right + 1) // 2:
        cl = parent*2+1; cr = cl+1
        child = cr if cr <= right and a[cr] > a[cl] else cl
        if temp >= a[child]: break
        a[parent] = a[child]; parent = child; BUILD[0] += 1
    a[parent] = temp

random.seed(2)
print("  n          이동횟수     n      n*log2(n)")
for n in (1000, 10000, 100000, 1000000):
    arr = [random.randint(0, 10**7) for _ in range(n)]
    BUILD[0] = 0
    for i in range((n - 1) // 2, -1, -1): dh_count(arr, i, n - 1)
    print(f"  {n:8d}  {BUILD[0]:10d}  {n:8d}  {n*math.log2(n):12.0f}")
print("\n→ 이동 횟수가 n log n 이 아니라 n 에 비례한다! (대략 n보다 조금 작음)")

---
# 💻 PART 6 — 코드 해부 (15~19번)

> 이제 개념은 다 잡았어. 실습 6-16의 **모든 줄**이 지금까지 배운 것 중 어디에 해당하는지 짚어보자.

### 15. 🟢 [설명] 두 함수의 역할 분담

```python
def heap_sort(a):
    def down_heap(a, left, right):      # ← 함수 A
        ...
    n = len(a)
    for i in range((n - 1) // 2, -1, -1):    # 1단계
        down_heap(a, i, n - 1)
    for i in range(n - 1, 0, -1):            # 2단계
        a[0], a[i] = a[i], a[0]
        down_heap(a, 0, i - 1)
```

**표를 채워봐.**

| | 하는 일 (한 줄) | 오늘 몇 번 문제에서 다뤘나 | 교재 몇 페이지 |
|---|---|---|---|
| `down_heap(a, left, right)` | ① | ② | 288~289p |
| `heap_sort` 1단계 | ③ | ④ | 292~293p |
| `heap_sort` 2단계 | ⑤ | ⑥ | 290~291p |

- 교재 295p: **"`down_heap`은 배열 a에서 `a[left]` ~ `a[right]` 원소를 힙으로 만듭니다. `a[left]` 이외에는 모두 힙 상태라고 가정하고 `a[left]`를 아랫부분의 알맞은 위치로 옮겨 힙 상태를 만듭니다."**
  → 이 **"가정"** 이 왜 중요해? 이 가정이 깨진 상태로 `down_heap`을 부르면 어떻게 될까?
- **1단계와 2단계가 같은 `down_heap`을 부르는데 인자가 달라.** 각각 `(i, n-1)`과 `(0, i-1)`이지. 뭐가 고정이고 뭐가 움직여?

*(답을 적은 뒤 실행)*

In [ ]:
# 가정이 깨진 상태로 down_heap을 부르면?
bad = [1, 3, 5, 7, 9, 2, 4, 6, 8, 10]     # 전혀 힙이 아닌 배열
t = bad[:]
down_heap(t, 0, len(t) - 1)               # 루트에만 down_heap
print("힙이 아닌 배열에 루트만 down_heap:")
print("  before:", bad)
print("  after :", t, "→ 힙?", is_heap(t)[0])
print("\n→ 아래가 힙이 아니면 down_heap 한 번으로는 절대 힙이 안 된다.")
print("   그래서 1단계에서 '아래에서 위로' 전부 훑는 것!")

# 1단계 인자 vs 2단계 인자
n = 10
print(f"\n[1단계] i가 {(n-1)//2} → 0 으로 감소, 호출은 down_heap(a, i, {n-1})")
print("   left가 움직이고 right는 고정 (범위는 항상 배열 전체)")
print(f"[2단계] i가 {n-1} → 1 로 감소, 호출은 down_heap(a, 0, i-1)")
print("   left는 0 고정, right가 줄어듦 (힙 범위가 점점 좁아짐)")

### 16. 🟡 [설명] `while parent < (right + 1) // 2` 의 정체

```python
parent = left
while parent < (right + 1) // 2:
```

이 조건이 뭘 뜻하는지 감이 잘 안 올 거야. 풀어보자.

- `parent`가 자식을 가지려면 **왼쪽 자식 인덱스 `2*parent+1`이 `right` 이하**여야 해. 부등식으로 쓰면:
```
2 * parent + 1 <= right
→ parent <= ①________
→ parent < ②________          (정수이므로)
```
- 정수 나눗셈으로 정리하면 `parent < (right + 1) // 2` 가 나와. 직접 유도해봐.
- 즉 이 조건은 **"③________________"** 라는 뜻이야.
- 그럼 조건이 **거짓**이 되는 순간은? `parent`가 어떤 노드일 때?

*(답을 적은 뒤 실행)*

In [ ]:
right = 9
print(f"right = {right} 일 때, 자식이 있는 노드는?")
print(f"  경계: (right+1)//2 = {(right+1)//2}")
for p in range(right + 1):
    cl = 2*p + 1
    has = cl <= right
    cond = p < (right + 1) // 2
    mark = "✅" if has == cond else "❌ 불일치!"
    print(f"  parent={p}: 왼쪽자식 {cl} {'존재' if has else '없음':4s} | "
          f"조건 p < {(right+1)//2} = {str(cond):5s} {mark}")

### 17. 🟡 [설명] `temp` — 교환이 아니라 "밀어내기"

교재 288~289p 설명은 계속 **"교환합니다"** 라고 하는데, 코드는 교환(swap)을 안 해.

```python
temp = a[left]              # ← 옮길 값을 잠시 빼둔다
parent = left
while ...:
    ...
    if temp >= a[child]:
        break
    a[parent] = a[child]    # ← 자식을 부모 자리로 '끌어올림' (한 방향 대입!)
    parent = child
a[parent] = temp            # ← 마지막에 한 번만 내려놓는다
```

**교환 버전과 비교해봐:**
```python
# 만약 매번 교환한다면
a[parent], a[child] = a[child], a[parent]
```

- 3단계 내려간다고 할 때, **대입 연산이 각각 몇 번** 일어나?
  - 교환 버전: ①____ 번
  - `temp` 버전: ②____ 번
- 왜 `temp` 방식이 더 효율적이지?
- 💡 이 패턴 어디서 봤지? **19일차 삽입 정렬**의 `tmp`가 정확히 같은 아이디어야:
```python
tmp = a[i]
while j > 0 and a[j-1] > tmp:
    a[j] = a[j-1]        # 한 칸씩 밀어내기
    j -= 1
a[j] = tmp
```
  → 두 코드의 **공통 구조**를 한 문장으로 정리해봐.

*(답을 적은 뒤 실행)*

In [ ]:
ASSIGN = [0]

def down_heap_swap(a, left, right):
    """매번 교환하는 버전"""
    parent = left
    while parent < (right + 1) // 2:
        cl = parent*2+1; cr = cl+1
        child = cr if cr <= right and a[cr] > a[cl] else cl
        if a[parent] >= a[child]: break
        a[parent], a[child] = a[child], a[parent]
        ASSIGN[0] += 2                     # 교환 = 대입 2번
        parent = child

def down_heap_temp(a, left, right):
    """temp를 쓰는 버전 (교재)"""
    temp = a[left]; ASSIGN[0] += 1         # temp에 저장
    parent = left
    while parent < (right + 1) // 2:
        cl = parent*2+1; cr = cl+1
        child = cr if cr <= right and a[cr] > a[cl] else cl
        if temp >= a[child]: break
        a[parent] = a[child]; ASSIGN[0] += 1
        parent = child
    a[parent] = temp; ASSIGN[0] += 1

random.seed(4)
data = [random.randint(0, 10**6) for _ in range(20000)]
for name, dh in (("교환 버전", down_heap_swap), ("temp 버전(교재)", down_heap_temp)):
    a = data[:]; ASSIGN[0] = 0
    n = len(a)
    for i in range((n-1)//2, -1, -1): dh(a, i, n-1)
    for i in range(n-1, 0, -1):
        a[0], a[i] = a[i], a[0]; dh(a, 0, i-1)
    print(f"{name:16s}: 대입 {ASSIGN[0]:9,d}회 | 정렬 {'OK' if a == sorted(data) else 'X'}")

### 18. 🟡 [빈칸] `down_heap` 직접 구현

7~9번, 16~17번에서 뜯어본 걸 조립해.

**기대 출력**
```
[9, 8, 5, 7, 3, 2, 4, 6, 1]
힙인가? True
```

In [ ]:
def my_down_heap(a: MutableSequence, left: int, right: int) -> None:
    """a[left]~a[right]를 힙으로 만들기 (a[left] 이외는 이미 힙이라고 가정)"""
    temp = a[left]                       # 옮길 값을 빼둔다

    parent = left
    while ___:                           # ① 자식이 있는 동안
        cl = ___                         # ② 왼쪽 자식
        cr = ___                         # ③ 오른쪽 자식
        child = cr if ___ and ___ else cl   # ④⑤ 오른쪽 자식이 존재하고 더 클 때
        if ___:                          # ⑥ 멈춰야 할 조건
            break
        a[parent] = ___                  # ⑦ 자식을 부모 자리로 끌어올림
        parent = ___                     # ⑧
    a[parent] = temp                     # 마지막에 한 번만 내려놓는다


# 7번 문제와 같은 상황: 루트를 1로 바꾼 힙
t = [1, 9, 5, 8, 3, 2, 4, 6, 7]
my_down_heap(t, 0, len(t) - 1)
print(t)
print("힙인가?", is_heap(t)[0])

### 19. 🟡 [빈칸] `heap_sort` 완성

15번에서 정리한 2단계를 코드로.

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
랜덤 500회 검증: 실패 0회
```

In [ ]:
def my_heap_sort(a: MutableSequence) -> None:
    """힙 정렬"""
    n = len(a)

    # 1단계: 배열 전체를 힙으로 만든다
    for i in range(___, ___, ___):       # ①②③ 어디서 어디까지, 어느 방향?
        my_down_heap(a, ___, ___)        # ④⑤

    # 2단계: 루트(최댓값)를 꺼내며 정렬한다
    for i in range(___, ___, ___):       # ⑥⑦⑧
        a[0], a[i] = ___, ___            # ⑨⑩ 최댓값과 맨 끝을 교환
        my_down_heap(a, ___, ___)        # ⑪⑫ 좁아진 범위를 다시 힙으로


x = [5, 8, 4, 2, 6, 1, 3, 9, 7, 0, 3, 5]
my_heap_sort(x)
print(x)

fail = 0
for _ in range(500):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    my_heap_sort(t)
    if t != ref: fail += 1
print(f"랜덤 500회 검증: 실패 {fail}회")

---
# 🎁 PART 7 — `heapq`로 마무리 (20~21번)

> 교재 296p 보충수업 6-5. 오늘 고생해서 만든 걸 **파이썬은 이미 갖고 있어.**
> 23일차에 잠깐 등장했던 `heapq.merge()`의 그 `heapq` 맞아.

### 20. 🟢 [실험] `heapq`는 최소 힙이다

교재 실습 6C-5:
```python
def heap_sort(a):
    heap = []
    for i in a:
        heapq.heappush(heap, i)      # 전부 밀어넣고
    for i in range(len(a)):
        a[i] = heapq.heappop(heap)   # 하나씩 꺼낸다
```

**여기서 이상한 점을 찾아봐.**

- 우리가 오늘 만든 힙은 **부모 ≥ 자식**(최대 힙)이라 루트가 **최댓값**이었어.
- 그런데 이 코드는 `heappop`을 순서대로 하면 **오름차순**이 나와. 그럼 `heapq`의 루트는 뭐지?
- 그럼 `heapq`는 **최대 힙**일까 **최소 힙**일까? ①________
- 그래서 우리 코드처럼 "뒤에서부터 채우기"가 아니라 **앞에서부터 채워도** 오름차순이 되는 거야.
- 💡 `heapq`로 **내림차순** 정렬을 하려면? (두 가지 방법이 있어 — 값에 `-`를 붙이거나, 꺼낸 걸 뒤집거나)

*(답을 적은 뒤 실행)*

In [ ]:
h = []
for v in [6, 4, 3, 7, 1, 9, 8]:
    heapq.heappush(h, v)
    print(f"push {v} → 내부 리스트 {h}")

show_tree(h, title="\nheapq 내부 구조를 트리로 보면")
print("루트 h[0] =", h[0], "← 최솟값!")
print("부모 <= 자식? (최소 힙 검사):",
      all(h[p] <= h[c] for p in range(len(h)) for c in (2*p+1, 2*p+2) if c < len(h)))

print("\npop 순서:", [heapq.heappop(h) for _ in range(7)])

print("\n[내림차순 만들기]")
data = [6, 4, 3, 7, 1, 9, 8]
h2 = []
for v in data: heapq.heappush(h2, -v)               # 부호 뒤집기
print("  방법1 (부호 반전):", [-heapq.heappop(h2) for _ in range(len(data))])
h3 = list(data); heapq.heapify(h3)
print("  방법2 (꺼낸 뒤 뒤집기):", [heapq.heappop(h3) for _ in range(len(data))][::-1])

### 21. 🟢 [구현] `heapq` 버전 + 오늘의 총정리

**(1) 구현** — 교재 실습 6C-5를 완성해.

**기대 출력**
```
[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]
랜덤 300회 검증: 실패 0회
```

**(2) 총정리** — R-2에서 예측했던 걸 이제 채워봐.

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? |
|---|---|---|---|---|
| 선택 (19일) | O(n²) | O(n²) | O(1) | ❌ |
| 퀵 (21~22일) | O(n log n) | O(n²) | O(log n) | ❌ |
| 병합 (23일) | O(n log n) | O(n log n) | O(n) | ✅ |
| **힙 (오늘)** | ① | ② | ③ | ④ |

- ③이 왜 그렇지? 힙 정렬은 **배열 안에서** 자리를 바꿔가며 정렬하잖아 (in-place).
- ④는 실행 셀에서 직접 확인해봐. 왜 그럴까? (힌트: 루트와 **맨 끝** 원소를 교환하는데, 그 둘은 서로 멀리 떨어져 있지 — 21일차 퀵 정렬과 같은 이유)
- 💡 **힙 정렬의 진짜 강점**: 최악에도 O(n log n)을 보장하면서 추가 메모리가 O(1)이야. 병합은 메모리를 쓰고, 퀵은 최악을 보장 못 하지. **셋 중 유일하게 둘 다 만족**해.
- 그런데도 실무 기본값은 왜 퀵/Timsort일까? 실측을 보고 답해봐.

In [ ]:
def heap_sort_heapq(a: MutableSequence) -> None:
    """힙 정렬 (heapq.heappush와 heapq.heappop을 사용)"""
    heap = []
    for i in a:
        ___                              # ① 힙에 밀어넣기
    for i in range(len(a)):
        a[i] = ___                       # ② 꺼내서 앞에서부터 채우기


x = [5, 8, 4, 2, 6, 1, 3, 9, 7, 0, 3, 5]
heap_sort_heapq(x)
print(x)

fail = 0
for _ in range(300):
    n = random.randint(0, 60)
    t = [random.randint(0, 40) for _ in range(n)]
    ref = sorted(t)
    heap_sort_heapq(t)
    if t != ref: fail += 1
print(f"랜덤 300회 검증: 실패 {fail}회")

In [ ]:
# ---- 안정성 확인 ----
def heap_sort_key(a):
    def d(a, left, right):
        temp = a[left]; parent = left
        while parent < (right + 1) // 2:
            cl = parent*2+1; cr = cl+1
            child = cr if cr <= right and a[cr][0] > a[cl][0] else cl
            if temp[0] >= a[child][0]: break
            a[parent] = a[child]; parent = child
        a[parent] = temp
    n = len(a)
    for i in range((n-1)//2, -1, -1): d(a, i, n-1)
    for i in range(n-1, 0, -1):
        a[0], a[i] = a[i], a[0]; d(a, 0, i-1)

base = [(3,'a'), (1,'b'), (3,'c'), (1,'d'), (2,'e'), (3,'f')]
t = base[:]; heap_sort_key(t)
print("입력      :", [x[1] for x in base], [x[0] for x in base])
print("힙 정렬   :", [x[1] for x in t], [x[0] for x in t])
print("sorted()  :", [x[1] for x in sorted(base, key=lambda z: z[0])], "← 안정")

# ---- 이번 주 전체 성능 ----
def hs_heapq(a):
    heap = []
    for i in a: heapq.heappush(heap, i)
    for i in range(len(a)): a[i] = heapq.heappop(heap)

def selection_sort(a):
    n = len(a)
    for i in range(n-1):
        m = i
        for j in range(i+1, n):
            if a[j] < a[m]: m = j
        a[i], a[m] = a[m], a[i]
def quick_sort(a):
    def qs(a, l, r):
        pl, pr = l, r; x = a[(l+r)//2]
        while pl <= pr:
            while a[pl] < x: pl += 1
            while a[pr] > x: pr -= 1
            if pl <= pr: a[pl], a[pr] = a[pr], a[pl]; pl += 1; pr -= 1
        if l < pr: qs(a, l, pr)
        if pl < r: qs(a, pl, r)
    if a: qs(a, 0, len(a)-1)
def merge_sort(a):
    def _ms(a, l, r):
        if l < r:
            c = (l+r)//2; _ms(a, l, c); _ms(a, c+1, r)
            p = j = 0; i = k = l
            while i <= c: buff[p] = a[i]; p += 1; i += 1
            while i <= r and j < p:
                if buff[j] <= a[i]: a[k] = buff[j]; j += 1
                else: a[k] = a[i]; i += 1
                k += 1
            while j < p: a[k] = buff[j]; k += 1; j += 1
    n = len(a); buff = [None]*n; _ms(a, 0, n-1)

random.seed(7)
N = 3000
data = [random.randint(0, 10**6) for _ in range(N)]
print(f"\n[n = {N} 랜덤]")
for nm, f in (("선택(19일)", selection_sort), ("퀵(21일)", quick_sort),
              ("병합(23일)", merge_sort), ("힙-직접(오늘)", heap_sort),
              ("힙-heapq(오늘)", hs_heapq), ("내장 sort", lambda x: x.sort())):
    a = data[:]
    t0 = time.perf_counter(); f(a); el = time.perf_counter() - t0
    print(f"  {nm:16s}: {el:.4f}s  {'OK' if a == sorted(data) else 'X'}")

---
---

# ✅ 정답 & 해설

> ⚠️ **트리를 직접 그려본 뒤에 내려와.** 특히 7·10·13번은 손으로 그려야 남아.

---

## 🔁 Remind

### R-1
- 안쪽 `for`는 **남은 구간 전체를 훑어 최솟값(또는 최댓값)의 위치를 찾아.** 라운드마다 `n-1, n-2, …, 1`번 비교 → 총 `n(n-1)/2`번 → **O(n²)**.
- 거의 정렬된 데이터를 줘도 빨라지지 않는 이유: **데이터 상태와 무관하게 항상 전체를 훑기 때문**. 삽입 정렬은 `while` 조건이 즉시 거짓이 되어 멈출 수 있지만, 선택 정렬엔 그런 탈출구가 없어.
- 💡 그래서 오늘의 목표: **"최댓값 찾기"를 O(n)에서 줄이는 것.**

### R-2
| 항목 | 병합 (23일) | 힙 (오늘) |
|---|---|---|
| 추가 메모리 | O(n) | ① **O(1)** |
| 안정성 | ✅ | ② **❌** |
| 최악 시간 | O(n log n) | ③ **O(n log n)** |

힙 정렬은 **배열 안에서 자리만 바꾸므로(in-place)** 추가 메모리가 상수야. 대신 멀리 떨어진 원소를 교환해서 안정성을 잃어. 21번에서 실증해.

---

## 🌳 PART 1 해설

### 1. 힙의 두 조건

- **조건 A (모양)**: ① **완전 이진 트리** — 위에서 아래로, 각 단계는 **왼쪽 자식부터** 채워져 빈틈이 없어야 함
- **조건 B (값)**: ② **모든 부모-자식 쌍에서 `부모의 값 ≥ 자식의 값`**

- 교재 286p: **"부모의 값이 자식의 값보다 항상 작아도 힙이라고 합니다. 즉, 이러한 두 값의 대소 관계가 일정하면 됩니다."** → 부모 ≥ 자식(**최대 힙**)이든 부모 ≤ 자식(**최소 힙**)이든 상관없어. **일관성**만 있으면 돼. (20번의 `heapq`가 후자!)
- 가장 큰 값은 **루트(a[0])** 에 있어. 교재 286p: "트리의 가장 위쪽에 위치한 루트가 가장 큰 값이 됩니다."
- **가장 작은 값은 특정할 수 없어.** 어딘가의 리프에 있다는 것만 알아. → 3번의 "부분 순서 트리"

---

### 2. 힙 판별

| | 힙? | 이유 |
|---|---|---|
| (가) `[9,7,8,6,5]` | ✅ | 모양 OK, 9≥7·8 / 7≥6·5 |
| (나) `[4,6,8,7,5,9]` | ❌ **값** | 4 < 6, 4 < 8 — 부모가 자식보다 작음 (교재 그림 6-32 ⓐ) |
| (다) `[9,7,8,6,5,4]` | ✅ | (나)를 힙으로 만든 것 (교재 그림 6-32 ⓑ) |
| (라) | ❌ **모양** | 8의 **왼쪽 자식이 비어 있는데 오른쪽에만** 자식이 있음 → 완전 이진 트리 아님 |
| (마) `[9,3,8,1,2,7,6]` | ✅ | 9≥3·8 / 3≥1·2 / 8≥7·6 — 형제끼리 3 < 8이어도 **상관없음!** |

**(라)가 핵심이야.** 값 조건은 다 맞는데 **모양**에서 탈락해. 교재 286p: **"'완전'은 부모는 왼쪽 자식부터 추가하여 모양을 유지한다는 뜻입니다."**
→ 이 모양 조건이 깨지면 **배열로 저장할 수 없어**. 5~6번에서 이유가 드러나.

**(마)** 는 3번(부분 순서 트리)의 예고편이야. 형제 `3`과 `8`이 뒤죽박죽인데도 힙이지.

---

### 3. 부분 순서 트리

- 형제 `8`과 `3`: 작은 쪽(3)이 **오른쪽**
- 형제 `6`과 `7`: 작은 쪽(6)이 **왼쪽**
- 두 답이 다르지? 그래서 **부분 순서 트리(partial ordered tree)**. 교재 287p 그대로야.

**실행 결과**
```
힙인가? True
배열   : [10, 9, 5, 8, 3, 2, 4, 6, 7, 1]
정렬됨? False        ← 힙이어도 정렬은 안 되어 있다!
최댓값은 항상 a[0] = 10   ← 이것만 O(1)에 보장
최솟값은? 1 ... 어디 있는지 알 수 없다
```

> 🔑 **힙은 "정렬"이 아니라 "최댓값 하나에 대한 약속"이야.** 전체를 정렬하는 비용을 내지 않고 최댓값만 즉시 얻는 구조. 그래서 우선순위 큐(priority queue)의 표준 구현이기도 해.

---

### 4. 부모와 자식의 인덱스

```
· 부모       : a[ ① (i - 1) // 2 ]
· 왼쪽 자식  : a[ ② i * 2 + 1 ]
· 오른쪽 자식: a[ ③ i * 2 + 2 ]
```

| i | a[i] | 부모 | 왼쪽 자식 | 오른쪽 자식 |
|---|---|---|---|---|
| 3 | 8 | ④ a[1] ⑤ =9 | ⑥ a[7]=6 | ⑦ a[8]=7 |
| 2 | 5 | ⑧ a[0] ⑨ =10 | ⑩ a[5]=2 | ⑪ a[6]=4 |
| 4 | 3 | a[1]=9 | ⑫ a[9]=1 (있음) | ⑬ a[10] → **없음** (n=10) |

- **`i = 0`에 부모 공식을 넣으면** `(0-1)//2 = -1`. 파이썬의 음수 인덱스는 "뒤에서부터"라 **에러 없이 조용히 마지막 원소**를 가리켜. 22일차 5번에서 본 `(0 + -1)//2` 함정과 정확히 같은 계열이야. 그래서 루트는 항상 별도 처리.
- 자식이 없는 노드 판별: **`2*i+1 >= n`** 이면 자식 없음. 16번의 `while` 조건이 바로 이걸 정리한 거야.

---

## 🔄 PART 2 해설

### 5. 트리 → 배열

**(가)** `[10, 9, 5, 8, 3, 2, 4, 6, 7, 1]` — 교재 [그림 6-33] 그대로.

**(나)** `[7, 5, 6, 2, 4, 3]` — 힙 ✅ (7≥5·6, 5≥2·4, 6≥3)

- 노드 6개, 배열 6칸, **낭비 0**. 이게 완전 이진 트리를 고집하는 이유야.
- (라)처럼 중간이 비면 `[9, 8, 7, None, None, 6, 5]` 처럼 빈 칸을 넣어야 하고, **깊이가 깊어질수록 빈 칸이 기하급수로 늘어나.** 그럼 인덱스 공식은 유지되지만 메모리가 터지지.

> 🔑 **"완전 이진 트리 + 레벨 순회 저장"이 있어야 `2i+1` 공식이 성립한다.** 이 셋은 한 세트야.

---

### 6. 배열 → 트리

**(가)** `[9, 8, 6, 7, 5, 4, 2, 3, 1]`
```
        9
     /     \
    8       6
   / \     / \
  7   5   4   2
 / \
3   1
```
힙 ✅

**(나)** `[8, 7, 5, 3, 6, 2, 4]`
```
        8
     /     \
    7       5
   / \     / \
  3   6   2   4
```
힙 ✅ (8≥7·5, 7≥3·6, 5≥2·4) — 형제 3 < 6이지만 문제없어.

**(다)** 5칸 배열에서 `a[4]`의 부모 = `a[(4-1)//2] = a[1]`, `a[1]`의 자식 = `a[3]`, `a[4]`

---

## ⬇️ PART 3 해설

### 7. 그림 6-34 재현

- **ⓐ** ① = **1** (마지막 원소). `1`이 있던 자리는 **사라져** (트리 크기가 10 → 9)
- **ⓑ** ② = **9**, ③ = **1**
- **ⓒ** ④ = **8**, ⑤ = **1**
- **ⓓ** ⑥ = **7**, ⑦ = **1**

**최종 배열**: `[9, 8, 5, 7, 3, 2, 4, 6, 1]`

- **중간에 멈추는 경우**: 교재 289p — **"이동할 원소값보다 왼쪽과 오른쪽 두 자식이 작으면 더 이상 교환할 수 없으므로 그 시점에서 스캔을 종료합니다."** 코드의 `if temp >= a[child]: break` 가 그거야.
- `1`이 이동한 단계 수는 **3번**. 트리 높이가 4단계(0~3)니까 최대 3번 내려갈 수 있고, 딱 끝까지 갔어. 일반적으로 **최대 log₂n 번** — 이게 `down_heap`이 O(log n)인 근거야.

---

### 8. 왜 마지막 원소를 루트로

- **자식 중 큰 놈을 올리면**: 그 자식 자리가 비고 → 또 그 자식이 올라오고 → 결국 **트리 중간 어딘가에 구멍**이 남아. 그럼 "왼쪽부터 빈틈없이"라는 **완전 이진 트리 조건이 깨져** → 배열로 저장 불가!
- **마지막 원소를 쓰면**: 사라지는 자리가 **가장 마지막 리프**야. 배열로 치면 맨 뒤 한 칸이 줄어드는 것뿐이라 **모양이 완벽하게 유지**돼.
- **"1 이외의 원소는 힙 상태를 유지"**: 마지막 원소는 리프였으니 자식이 없고, 그 부모는 다른 자식과의 관계가 그대로야. 즉 **트리 전체에서 어긋난 곳은 새 루트 하나뿐.**
- → 그래서 고칠 곳이 딱 하나고, 그걸 아래로 흘려보내는 게 `down_heap`. **문제가 국소화(localized)되는 것**이 이 설계의 핵심이야.

---

### 9. 왜 큰 자식과 교환

- **작은 자식(5)과 교환하면**: 새 루트가 `5`인데 왼쪽 자식이 `9` → `5 < 9`로 **바로 힙이 깨져.** 실행 결과에서 `a[0]=5 < a[1]=9` 로 확인돼.
- **큰 자식(9)과 교환하면**: `9`는 두 자식 중 최댓값이니 나머지 자식(5)보다도 크고, 내려간 값보다도 큼. → **루트 자리는 확실히 해결**되고, 문제가 한 단계 아래로만 밀려나.

```python
child = cr if cr <= right and a[cr] > a[cl] else cl
```
- 말로 풀면: **"오른쪽 자식이 존재하고(`cr <= right`) 왼쪽 자식보다 크면 오른쪽을, 아니면 왼쪽을 고른다."**
- `cr <= right`가 필요한 이유: 완전 이진 트리에서 **왼쪽 자식만 있고 오른쪽은 없는 노드가 딱 하나** 있을 수 있어. 이 검사 없이 `a[cr]`을 읽으면 범위 밖 또는 **이미 정렬된 구간의 값**을 잘못 읽어.
- 두 자식이 같으면 `>` 가 거짓이라 **왼쪽(`cl`)** 이 선택돼. 값이 같으니 힙 조건엔 문제없지만, **동점 처리 순서가 정해진다는 뜻**이라 안정성과 얽혀. (21번)

---

## 🔁 PART 4 해설

### 10. 그림 6-35 재현

- **ⓐ** ① = **1**, ② = **10** → `[1, 9, 5, 8, 3, 2, 4, 6, 7, 10]`, 정렬완료 `[10]`
- **ⓑ** 재구성 결과 트리의 ③ = **9**, ④ = **8** (7번 결과와 동일: `[9,8,5,7,3,2,4,6,1]`)
  교환 후 ⑤ = **1** → `[1, 8, 5, 7, 3, 2, 4, 6, 9, 10]`, 정렬완료 `[9, 10]`
- **ⓒ** 정렬완료 `[8, 9, 10]`
- **ⓓ** `[7, 8, 9, 10]`
- **ⓔ** `[6, 7, 8, 9, 10]`

실행 결과와 대조해봐. 배열 뒤쪽이 **10 → 9,10 → 8,9,10** 순으로 자라나지.

---

### 11. 4단계 알고리즘

```
1. i값을 ① n - 1 로 초기화합니다.
2. ② a[0]과 a[i] 를 교환합니다.
3. a[0], a[1], ..., a[i-1]을 ③ 힙으로 만듭니다.
4. i값을 ④ 1씩 감소 시켜 0이 되면 종료합니다. 그렇지 않으면 ⑤ 2 번으로 돌아갑니다.
```

파이썬으로:
```python
for i in range(n - 1, 0, -1):
    a[0], a[i] = a[i], a[0]
    down_heap(a, 0, i - 1)
```

- 힙 정렬은 총 **2단계**: **① 배열을 힙으로 만들기**(PART 5) + **② 루트를 꺼내며 정렬하기**(여기).

---

### 12. 왜 뒤에서부터 자랄까

- 루트는 **최댓값**. 오름차순 정렬에서 최댓값이 갈 자리는 **배열의 맨 끝**이야.
- "꺼낸다" 대신 **"맨 끝과 교환한다"** 를 쓰는 이유: 꺼낸 값을 담을 **별도의 배열이 필요 없어져.** 맨 끝 원소는 어차피 루트로 올려야 하니까(8번), 한 번의 교환으로 **두 가지 일이 동시에** 처리돼. → 추가 메모리 O(1)!

| | 매 라운드 | 최댓값 찾기 | 라운드 수 | 전체 |
|---|---|---|---|---|
| 선택 정렬 | 남은 구간 전체를 훑어 최댓값 | ① **O(n)** | ② **n** | ③ **O(n²)** |
| 힙 정렬 | 루트가 곧 최댓값, 꺼낸 뒤 재구성 | ④ **O(log n)** | ⑤ **n** | ⑥ **O(n log n)** |

- **`down_heap`이 O(log n)인 이유**: 한 단계 내려갈 때마다 **남은 서브트리 크기가 절반**이 돼. 완전 이진 트리의 높이가 log₂n이니 최대 log₂n번 내려가.
- **이진 검색과 비슷한 점** (교재 296p): 매번 후보 범위가 절반으로 줄어드는 것. 12일차 이진 검색에서 배운 그 구조야.

**실측**
```
n        총 이동단계   n*log2(n)
    100         480          664
   1000        8048         9966
  10000      113997       132877
 100000     1474920      1660964
```
총 이동 단계가 **n log n에 비례**하며 자라지 (계수가 약 0.9).

---

## 🏗️ PART 5 해설

### 13. 그림 6-37 재현

- **ⓐ** ① = **10**, ② = **9** → `[1, 3, 5, 7, 10, 2, 4, 6, 8, 9]`
- **ⓑ** ③ = **8**, ④ = **8**, ⑤ = **7** → `[1, 3, 5, 8, 10, 2, 4, 6, 7, 9]`
- **ⓒ** ⑥ = **변화 없음** (5 ≥ 2, 4 — 이미 힙)
- **ⓓ** ⑦ = **10** → 3이 내려가고 또 한 번 더 내려감 → ⑧ = **10**, ⑨ = **3**
  → `[1, 10, 5, 8, 9, 2, 4, 6, 7, 3]`
- **ⓔ** 루트 1을 흘려보내면 → `[10, 9, 5, 8, 3, 2, 4, 6, 7, 1]`

🎯 **5번 문제의 트리(그림 6-33)와 정확히 같아져!** 교재가 두 그림을 연결해둔 거야.

---

### 14. `(n-1)//2` 부터 역순

**(1) 시작점**
- n=10일 때 `(10-1)//2 = 4`. `a[4]`의 왼쪽 자식은 `a[9]` → 존재. `a[5]`의 왼쪽 자식은 `a[11]` → 없음.
- 즉 **`(n-1)//2`는 자식을 가진 마지막 노드**야. 그 뒤는 전부 리프이고, **리프는 그 자체로 이미 힙**(자식이 없으니 조건 위반 불가)이라 손댈 필요가 없어.
- `n//2 - 1` 과 비교: n=10이면 `4` (같음), **n=11이면 `(11-1)//2 = 5` vs `11//2-1 = 4`** → **다름!** 홀수일 때 `n//2-1`은 한 칸 덜 커버해서 틀려. 교재 공식이 안전해.

**(2) 왜 역순인가**
- `down_heap`은 **"`a[left]` 아래는 이미 힙"** 을 가정해. 정순(`i=0`부터)이면 이 가정이 성립하지 않아 — 아래가 아직 엉망이니까.
- 아래에서 위로 올라가면, `a[i]`를 처리하는 시점에 **`a[i]`의 두 서브트리는 이미 힙 처리가 끝난 상태**야. 가정이 항상 만족돼.
- **실측**: 역순 → `[10,9,5,8,3,2,4,6,7,1]` 힙 ✅ / 정순 → `[5,9,4,8,10,2,1,6,7,3]` 힙 ❌

**(3) 시간 복잡도 함정** 🔥
```
n          이동횟수      n*log2(n)
   1000        728          9966
  10000       7433        132877
 100000      74309       1660964
1000000     742940      19931569
```
이동 횟수가 **정확히 n에 비례**(약 0.74n)해. `n log n`과는 자릿수가 달라.

**왜?** 노드의 절반은 리프(0단계 이동), 4분의 1은 최대 1단계, 8분의 1은 최대 2단계… 합치면
`n/2·0 + n/4·1 + n/8·2 + … ≈ n` 으로 수렴해. **깊이가 깊은 노드는 개수가 적기 때문**이야.

> 🔑 **힙 만들기는 O(n), 정렬은 O(n log n).** 전체는 O(n log n)이지만, "힙을 만드는 것 자체"는 놀랍도록 싸. 그래서 "최댓값 k개만 뽑기" 같은 작업에서 힙이 강력해.

---

## 💻 PART 6 해설

### 15. 두 함수의 역할

| | 하는 일 | 오늘 문제 | 교재 |
|---|---|---|---|
| `down_heap(a, left, right)` | ① `a[left]` 하나만 아래로 흘려보내 힙 상태 복구 | ② **7~9번** | 288~289p |
| 1단계 | ③ 무작위 배열을 힙으로 만들기 (상향식) | ④ **13~14번** | 292~293p |
| 2단계 | ⑤ 루트를 맨 끝과 교환하고 범위를 줄이며 반복 | ⑥ **10~12번** | 290~291p |

- **가정이 깨지면**: 실행 결과처럼 `down_heap` 한 번으로는 절대 힙이 안 돼. 아래가 힙이어야만 "한 원소만 고치면 된다"가 성립하거든.
- **인자 차이**:
  - 1단계 `down_heap(a, i, n-1)` — `left`가 움직이고 `right`는 **고정**(항상 배열 전체)
  - 2단계 `down_heap(a, 0, i-1)` — `left`는 0 **고정**, `right`가 줄어듦(힙 범위가 좁아짐)
  - → **같은 함수를 정반대 방식으로 재사용**하는 게 이 코드의 우아한 지점이야.

---

### 16. `while parent < (right + 1) // 2`

```
2 * parent + 1 <= right
→ 2 * parent <= right - 1
→ parent <= ① (right - 1) / 2
→ parent < ② (right + 1) / 2
```
정수 나눗셈으로 `parent < (right + 1) // 2`.

- 뜻: **③ "parent에게 자식이 하나라도 있는가"**
- 조건이 거짓이 되는 순간 = `parent`가 **리프**에 도달했을 때. 더 내려갈 곳이 없으니 종료.
- 실측에서 `right=9`일 때 경계가 `5`이고, `parent = 0~4`만 자식을 갖는 것과 정확히 일치해.

---

### 17. `temp` — 밀어내기

**3단계 내려간다면**
- 교환 버전: 교환 3번 × 대입 2번 = ① **6번**
- `temp` 버전: `temp` 저장 1 + 끌어올림 3 + 마지막 안착 1 = ② **5번**

단계가 깊어질수록 차이가 커져 (k단계면 `2k` vs `k+2`).

**실측 (n=20,000)**
```
교환 버전       : 대입 496,720회
temp 버전(교재) : 대입 308,358회    ← 약 1.6배 적음
```

**왜 되는가**: 내려가는 값(`temp`)은 어차피 **최종 위치에 도달할 때까지 중간 자리에 있을 필요가 없어.** 중간 단계에서 `a[parent] = temp` 를 해봤자 다음 반복에서 다시 덮어써지거든. 그러니 **끝에 한 번만** 쓰면 돼.

**19일차 삽입 정렬과의 공통 구조**:
> **"이동할 값을 임시 변수에 빼두고, 나머지를 한 칸씩 밀어낸 뒤, 마지막에 한 번만 제자리에 놓는다."**

삽입 정렬은 **옆으로**(1차원), 힙 정렬은 **아래로**(트리) 밀어낸다는 것만 달라. 오늘 배운 게 완전히 새로운 게 아니라는 증거야.

---

### 18. `my_down_heap`

```python
while parent < (right + 1) // 2:                     # ①
    cl = parent * 2 + 1                              # ②
    cr = cl + 1                                      # ③
    child = cr if cr <= right and a[cr] > a[cl] else cl   # ④⑤
    if temp >= a[child]:                             # ⑥
        break
    a[parent] = a[child]                             # ⑦
    parent = child                                   # ⑧
```
출력: `[9, 8, 5, 7, 3, 2, 4, 6, 1]`, 힙 True

---

### 19. `my_heap_sort`

```python
for i in range((n - 1) // 2, -1, -1):    # ①②③
    my_down_heap(a, i, n - 1)            # ④⑤

for i in range(n - 1, 0, -1):            # ⑥⑦⑧
    a[0], a[i] = a[i], a[0]              # ⑨⑩
    my_down_heap(a, 0, i - 1)            # ⑪⑫
```
출력: `[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]`, 랜덤 500회 실패 0회

**주의할 두 곳**
- 1단계 `range((n-1)//2, -1, -1)` — 끝이 `-1`이어야 `i=0`(루트)까지 포함돼. `0`으로 쓰면 루트를 빠뜨려.
- 2단계 `range(n-1, 0, -1)` — 끝이 `0`이야. `i=0`이면 자기 자신과 교환하는 무의미한 동작이니 제외.

---

## 🎁 PART 7 해설

### 20. `heapq`는 최소 힙

- ① **최소 힙** (부모 ≤ 자식). 루트 `h[0]`이 **최솟값**이야.
- 그래서 `heappop`을 반복하면 작은 값부터 나오고, **앞에서부터 채워도** 오름차순이 완성돼.
- 우리가 만든 최대 힙은 루트가 최댓값이라 **뒤에서부터** 채웠지. 방향만 정반대야.

**실행 결과**
```
push 6 → [6]
push 4 → [4, 6]
push 3 → [3, 6, 4]        ← 3이 루트로 올라감
push 7 → [3, 6, 4, 7]
push 1 → [1, 3, 4, 7, 6]  ← 1이 루트로
...
pop 순서: [1, 3, 4, 6, 7, 8, 9]
```

**내림차순 만들기**
- 방법1: 넣을 때 `-v`, 꺼낼 때 `-heappop()` — 부호를 뒤집어 최대 힙처럼 쓰기
- 방법2: 전부 꺼낸 뒤 `[::-1]` 로 뒤집기

💡 `heapq`엔 `heapify(list)` 도 있어. 리스트를 **제자리에서 O(n)에** 힙으로 만들어 — 14번 (3)에서 본 그 O(n)이야.

---

### 21. `heapq` 버전 + 총정리

```python
heap = []
for i in a:
    heapq.heappush(heap, i)      # ①
for i in range(len(a)):
    a[i] = heapq.heappop(heap)   # ②
```
출력: `[0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9]`, 랜덤 300회 실패 0회

**총정리표**

| 정렬 | 평균 | 최악 | 추가 메모리 | 안정? |
|---|---|---|---|---|
| 선택 (19일) | O(n²) | O(n²) | O(1) | ❌ |
| 퀵 (21~22일) | O(n log n) | O(n²) | O(log n) | ❌ |
| 병합 (23일) | O(n log n) | O(n log n) | O(n) | ✅ |
| **힙 (오늘)** | ① **O(n log n)** | ② **O(n log n)** | ③ **O(1)** | ④ **❌** |

- ③ **O(1)인 이유**: 배열 안에서 교환만 하니까(in-place). 23일차 병합의 `buff` 같은 게 필요 없어.
- ④ **불안정한 이유**: 루트 `a[0]`과 **맨 끝 `a[i]`** 를 교환하는데, 이 둘은 배열에서 가장 멀리 떨어진 자리야. 같은 값 사이를 뛰어넘어 자리가 바뀌니 원래 순서가 보존될 수 없어. **21일차 퀵 정렬과 정확히 같은 이유**지.

**실측 (안정성)**
```
입력      : ['a','b','c','d','e','f'] [3,1,3,1,2,3]
힙 정렬   : ['d','b','e','c','f','a'] [1,1,2,3,3,3]   ← 순서 뒤죽박죽 ❌
sorted()  : ['b','d','e','a','c','f']                 ← 안정 ✅
```

**실측 (속도, n=3,000)**
```
선택(19일)     : 0.2071s
퀵(21일)       : 0.0032s
병합(23일)     : 0.0056s
힙-직접(오늘)   : 0.0068s
힙-heapq(오늘)  : 0.0011s
내장 sort      : 0.0004s
```

- 힙 정렬은 **선택 정렬 대비 약 30배** 빨라. "최댓값 찾기를 O(n) → O(log n)으로 줄인다"는 아이디어가 그대로 숫자로 나온 거야.
- 그런데 **퀵보다는 2배 느려.** 왜? 힙 정렬은 부모↔자식을 오가며 **인덱스가 널뛰기(2i+1)** 해서 CPU 캐시 적중률이 나빠. 퀵은 인접한 메모리를 순차적으로 훑지.
- 그래서 실무 기본값은 퀵/Timsort이고, **힙 정렬은 "최악을 반드시 막아야 할 때"** 쓰여. (C++의 `introsort`는 퀵으로 시작해서 재귀가 너무 깊어지면 **힙 정렬로 갈아타** — 최악 O(n²)를 O(n log n)으로 봉인하는 안전장치야.)

> 🔑 **힙 정렬은 유일하게 "최악 O(n log n) + 추가 메모리 O(1)"을 동시에 만족한다.** 병합은 메모리를 쓰고, 퀵은 최악을 보장 못 해. 셋 다 필요한 이유가 있는 거야.

---

## 📌 핵심 3줄 요약

1. **힙 정렬은 선택 정렬의 진화형이다.** 매 라운드 최댓값을 찾는 비용을 O(n)에서 O(log n)으로 줄인 것뿐인데, 전체가 O(n²) → O(n log n)이 됐어. 실측 30배 차이.
2. **"완전 이진 트리"라는 모양 조건이 배열 저장을 가능하게 한다.** 빈틈이 없으니 `부모 (i-1)//2`, `자식 2i+1 / 2i+2` 공식이 성립하고, 포인터도 노드 객체도 필요 없어. 힙의 마법은 값 조건이 아니라 **모양 조건**에서 나와.
3. **`down_heap` 하나로 두 단계를 다 한다.** 1단계는 `right` 고정 + `left` 이동(상향식 힙 만들기, O(n)), 2단계는 `left` 고정 + `right` 축소(정렬, O(n log n)). 같은 함수를 정반대로 재사용하는 게 이 코드의 설계야.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 4, 5, 7, 11, 15, 20번)**: 전원 필수
  - **2번 힙 판별**에서 (라)를 놓치면 안 돼 — "모양이 틀린 것"과 "값이 틀린 것"의 구분이 오늘의 출발점
  - **7번 그림 6-34**는 반드시 손으로 그릴 것. 이거 하나면 힙 정렬의 90%야
- 🟡 **(R-2, 3, 6, 8, 9, 10, 12, 13, 16, 17, 18, 19, 21번)**: 팀 목표선
  - **10번 + 13번**을 나란히 보면 "힙 만들기 → 정렬"의 전체 그림이 완성돼
  - **17번 `temp` 밀어내기**는 19일차 삽입 정렬과 연결되는 지점이라 꼭 짚고 갈 것
- 🔴 **(14번)**: 도전
  - **14번 (3)이 오늘 최고 난도** 🔥 — 힙 만들기가 O(n log n)이 아니라 **O(n)** 이라는 반전. 실측 데이터가 증거야
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 21번)

## 🔗 오늘 회수된 개념들

- **19일차 선택 정렬** → 힙 정렬의 출발점, "최댓값 찾기 비용 줄이기" (R-1, 12번)
- **19일차 삽입 정렬의 `tmp`** → `down_heap`의 `temp` 밀어내기 (17번)
- **12일차 이진 검색** → `down_heap`이 매번 범위를 절반으로 (12번)
- **21일차 퀵 정렬의 불안정성** → 힙도 멀리 떨어진 원소를 교환해서 불안정 (21번)
- **22일차 `(0-1)//2` 바닥 나눗셈 함정** → 루트의 부모 공식이 -1이 되는 문제 (4번)
- **23일차 `heapq.merge`** → 오늘 `heapq`의 정체가 밝혀짐 (20번)
- **23일차 병합 정렬** → 메모리 O(n) vs O(1), 안정 vs 불안정 대조 (R-2, 21번)

---

> **다음 진도 (25일차)**: 06-9 **도수 정렬(counting sort)** — 교재 297p.
> 오늘까지 배운 정렬은 **전부 원소끼리 비교**했어. 그런데 도수 정렬은 **비교를 아예 안 해.**
> 그래서 O(n log n)의 벽을 뚫고 **O(n)** 이 가능해져. 대신 조건이 붙지 — 어떤 조건일지 미리 생각해봐.